# 의도 분류 + 1차 검색 방식 비교 (Colab A100 · Qwen3.5-27B 분류 · 임직원 골든셋)

질문이 **의도 분류**(규칙 + 분류 보조 LLM, backend main 그대로)를 거쳐 검색까지 가는지, 가면 두 검색 방식 중 어느 쪽이 정답을 LLM 입력까지 가져오는지 봐요. 답변 생성은 안 해요.
분류 부분은 `colab/intent_search_eval.ipynb`(11/23)와 같은 코드예요 (vLLM으로 27B를 띄워 backend `llm_classify.chat_json`만 바꿔 끼움).

```
질문 → guard(마스킹) → classify (보안문서 판단 + 의도 분류, 분류 LLM = Qwen3.5-27B)
     → 검색으로 가나? (backend build.py run_pipeline 분기)
          sensitive · 대화기록 거절 · 기밀문서 · refuse · no_question · out_of_scope · lookup → 검색 안 함
          rule · both → 검색 (아래 두 방식)
```

| 방식 | 1차 검색 대상 | 리랭커에 넣는 것 | LLM에 넘기는 5개 |
| --- | --- | --- | --- |
| **① 조+전결 → 펼치기** | 조문 단위 조각(조문·별표·서식안내·부칙) **+ 전결항목**(DB #25) | 후보 조를 **항으로 펼치고**, 연결된 **별표**(`appendix_links`)·**서식안내**(`form_links`), **인용된 조**(`article_refs` 따른다·준용, 예외는 반대 방향)를 더함. 별표당 3조각·전체 200쌍까지(임베딩 유사도 순) | 조마다 항 1,200자 + 별표 2조각 + 서식안내 1 |
| **② 모든 조각 → 리랭크 (40)** | 항 단위 검색 조각 전부(항·별표·서식안내·부칙·전결항목) | 1차 상위 40조각 그대로 | 조(article_id)로 묶어 5개, 묶음마다 1,200자 |

검색 공통: Dense(BGE-M3) + BM25(Kiwi, backend `rewrite`) → RRF(법령·내부규정 같은 점수, 가중치 없음) → bge-reranker-v2-m3, 내부규정 가점 0.

**지표** (임직원 골든셋 dev·test, 정답 조각이 있는 문항. uncertain은 뺌)
- **검색 누락률**: 정답이 있는데 분류 때문에 검색을 안 한 비율 (두 방식 같음)
- **2단계 종합**: 검색 안 한 문항은 0으로 셈 → 사용자가 실제로 받는 품질. 정답 조각 recall@5
- **분류 없이 검색만**: 같은 문항을 분류를 건너뛰고 모두 검색했을 때. 이것 − 2단계 종합 = 분류 때문에 잃은 만큼
- **정답 조각**(`gold_hang.primary`, 항·별표 조각 id) 기준
  - recall@5: LLM에 넘긴 5묶음에 정답 조각이 **잘리지 않고** 들어간 비율 (본문 1,200자·별표 1,500자 안)
  - recall@20 · MRR: 리랭커 점수 순으로 줄 세운 **조각** 목록에서 정답 조각의 순위
  - (참고) 조문 recall@5: 예전 지표 (`gold_articles`)
- LLM 입력 글자수, 리랭커 쌍 수, 분류·검색 시간 p50·p95, ①에서 인용·예외로만 정답이 들어온 문항

런타임 → 런타임 유형 변경 → **A100**(40GB↑, 27B FP8) 또는 H100 → 모두 실행. 27B 다운로드·vLLM 설치는 검색 비교를 도는 동안 백그라운드로 해요.

## 1. 설정 · 저장소 받기

In [ ]:
!pip install -q -U "sentence-transformers>=3.4" rank_bm25 kiwipiepy python-dotenv openai

In [ ]:
import os, sys, json, re, time, hashlib, shutil, importlib.util, subprocess
from datetime import date, datetime
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch

ROOT = Path(os.environ.get("LOCAL_ROOT", "/content/hr_chatbot"))   # LOCAL_ROOT: 내 컴퓨터에서 시험할 때만
ROOT.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path(os.environ.get("LOCAL_ROOT", "/content")) / "first_stage_results" / f"first_stage_{datetime.now():%Y%m%d_%H%M}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {# 분류 보조 LLM (intent_search_eval.ipynb 와 같음)
       "llm_model": "auto",            # auto: 75GB↑ Qwen/Qwen3.5-27B · 39GB↑ -FP8. 27B가 안 들어가면 멈춤
       "served_name": "qwen3.5-27b", "llm_port": 8000, "max_model_len": 8192,
       "vllm_gpu_util": 0.75,          # 같은 GPU에 BGE-M3·리랭커가 같이 있어서 낮춤
       "employee_login": "E1584",      # 페르소나 없는 문항용 (backend scripts/eval_intent.py)
       "embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
       "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
       "bm25_weight": 1.0, "dense_weight": 1.0, "rewrite_bm25": True,
       "k_context": 5, "max_chars_per_article": 1200,
       # ① 펼치기 상한
       "ref_top_units": 10, "ref_per_unit": 3, "appx_chunks": 3, "pair_cap": 200,
       # LLM에 붙이는 하위 항목
       "ctx_appx": 2, "ctx_form": 1, "max_chars_appx": 1500,
       # ② 1차 후보 수
       "k_all_small": 40, "k_all_big": 200}

if "LOCAL_ROOT" not in os.environ:
    for repo in ("backend", "DB", "RAG"):
        d = ROOT / repo
        if d.exists():
            subprocess.run(f"rm -rf {d}", shell=True)
        subprocess.run(f"git clone -q --depth 1 https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git {d}", shell=True, check=True)
COMMITS = {repo: subprocess.run(f"git -C {ROOT / repo} log -1 --format='%h %ad %s' --date=short", shell=True,
                               capture_output=True, text=True).stdout.strip() for repo in ("backend", "DB", "RAG")}
for repo, c in COMMITS.items():
    print(f"{repo:8s} {c}")
BACKEND, DB, HYERIM = ROOT / "backend", ROOT / "DB", ROOT / "RAG"

# 팀 채점 스크립트 (골든셋·dev/test 분할)
os.environ["BACKEND_DIR"] = str(BACKEND)
spec = importlib.util.spec_from_file_location("eval_golden", HYERIM / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eg)
ITEMS = eg.load_items()
SPLIT = eg.get_splits(ITEMS)

# backend 검색어 보강(glossary). DB 없이 불러옴
os.environ.update({"HRDB_URL": "", "CHAT_LOG_DB": "false", "LLM_BASE_URL": "", "CLASSIFY_LLM": "off",
                   "SEARCH_MODE": "keyword", "PSEUDO_KEY": hashlib.sha256(os.urandom(16)).hexdigest()})
sys.path.insert(0, str(BACKEND))
from app import config as be_config
from app.graph import nodes as be_nodes, llm_classify, admin_intent_llm, admin_params_llm, conf_doc
from app.graph import retriever as be_retriever
print(f"골든셋 {len(ITEMS)}문항 · 재작성 예: {be_retriever.rewrite('결혼하면 경조휴가 며칠이야?')}")

## 2. GPU 확인 · 분류 모델 고르기

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("GPU가 없어요. 런타임 → 런타임 유형 변경 → A100 또는 H100으로 바꾸고 다시 실행하세요.")
lines = gpu.splitlines()
GPU_NAME, N_GPU = lines[0].split(",")[0].strip(), len(lines)
TOTAL_GB = sum(int(l.split(",")[1]) for l in lines) / 1024
if CFG["llm_model"] == "auto":
    if TOTAL_GB >= 75:   CFG["llm_model"] = "Qwen/Qwen3.5-27B"
    elif TOTAL_GB >= 39: CFG["llm_model"] = "Qwen/Qwen3.5-27B-FP8"
    else:
        raise RuntimeError(f"{GPU_NAME} {TOTAL_GB:.0f}GB로는 Qwen3.5-27B가 안 들어가요 (L4 등). A100(40GB↑) 또는 H100으로 바꾸세요.")
print(f"GPU {GPU_NAME} × {N_GPU}, {TOTAL_GB:.0f}GB → 분류 LLM {CFG['llm_model']}")

## 3. vLLM 설치 · 27B 다운로드 (백그라운드)

GPU는 안 써요. 시작해 두고 아래 셀을 계속 실행하세요 (그 사이 검색 비교를 먼저 돌려요).

In [ ]:
VLLM_ENV = Path("/content/vllm-env")
SETUP_LOG = Path("/content/llm_setup.log")
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""      # 보안 비밀(열쇠 아이콘)에 있으면 다운로드가 빨라요. 없어도 됨
except Exception:
    pass
setup_script = f"""set -e
echo "[1/2] vLLM 설치 $(date)"
if [ ! -x {VLLM_ENV}/bin/vllm ]; then
  pip install -q uv
  rm -rf {VLLM_ENV}
  uv venv {VLLM_ENV} --python 3.12 -q
  uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.17.0" --torch-backend=auto
fi
echo "[2/2] 모델 다운로드 {CFG['llm_model']} $(date)"
{"export HF_TOKEN=" + HF_TOKEN if HF_TOKEN else ""}
{VLLM_ENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{CFG['llm_model']}')"
echo "SETUP_DONE $(date)"
"""
Path("/content/llm_setup.sh").write_text(setup_script)
if "SETUP_DONE" not in (SETUP_LOG.read_text() if SETUP_LOG.exists() else ""):
    SETUP_PROC = subprocess.Popen(f"bash /content/llm_setup.sh > {SETUP_LOG} 2>&1", shell=True)
    print("백그라운드 시작. 진행 상황:", SETUP_LOG)
else:
    print("이미 설치·다운로드됨")

## 4. 데이터 · 연결표 불러오기

In [ ]:
def jl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def gold_of(it, key): return (it.get(key) or {}).get("primary") or []
def squash(s): return re.sub(r"\s+", "", s or "")
def dedupe(xs): return list(dict.fromkeys(xs))

def as_chunk(r):
    return {"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
            "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
            "type": r.get("chunk_type", ""), "article": (r.get("article") or "").strip(),
            "article_title": (r.get("article_title") or "").strip(), "text": r["text"],
            "article_id": r.get("article_id"),
            "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]}

# 항 단위 검색 조각 전부 (②의 1차 대상, ①의 리랭크 대상)
NEWC = [as_chunk(r) for r in jl(DB / "data/hang/chunks.jsonl") if r.get("searchable")]
NEW_POS = {c["id"]: j for j, c in enumerate(NEWC)}
NEW_BY_ART = defaultdict(list)
for j, c in enumerate(NEWC):
    NEW_BY_ART[c["article_id"]].append(j)

# ①의 1차 대상: 조문 단위 조각(예전 그대로) + 전결항목
O2N = json.load(open(DB / "data/hang/old_to_new.json", encoding="utf-8"))
M1C = [as_chunk(r) for p in ("data/regs/chunks.jsonl", "data/laws/chunks.jsonl") for r in jl(DB / p) if r.get("searchable")]
for c in M1C:
    c["article_id"] = O2N[c["id"]]["article_id"]
M1C += [c for c in NEWC if c["type"] == "전결항목"]

ARTICLES = {a["article_id"]: a for a in jl(DB / "data/hang/articles.jsonl")}      # 조 (별표·부칙 없음)
AL = json.load(open(DB / "data/hang/appendix_links.json", encoding="utf-8"))
FL = json.load(open(DB / "data/hang/form_links.json", encoding="utf-8"))
APPX_OF, ART_OF_APPX = AL["article_to_appendix"], AL["appendix_to_article"]
FORM_OF, ART_OF_FORM = FL["article_to_form"], FL["form_to_article"]
REFS = json.load(open(DB / "data/hang/article_refs.json", encoding="utf-8"))["refs"]
REF_FWD, EXC_REV = defaultdict(list), defaultdict(list)
for r in REFS:
    if r["kind"] in ("따른다", "준용"):
        REF_FWD[r["from"]].append(r["to"])
    elif r["kind"] == "예외":
        EXC_REV[r["to"]].append(r["from"])    # 기본 조(to)가 뽑히면 예외 조(from)를 더함
REF_FWD = {k: dedupe(v) for k, v in REF_FWD.items()}
EXC_REV = {k: dedupe(v) for k, v in EXC_REV.items()}

print(f"① 1차 대상 {len(M1C)}조각 ({dict(Counter(c['type'] for c in M1C))})")
print(f"② 1차 대상 {len(NEWC)}조각 ({dict(Counter(c['type'] for c in NEWC))})")
print(f"조 {len(ARTICLES)} · 별표 연결 {sum(map(len, APPX_OF.values()))} · 서식 연결 {sum(map(len, FORM_OF.values()))} · "
      f"인용(따른다·준용) {sum(map(len, REF_FWD.values()))} · 예외 {sum(map(len, EXC_REV.values()))}")

# 골든셋 정답 id 가 지금 데이터에 있는지 (DB 가 바뀌면 어긋날 수 있음)
miss_art = Counter(a for it in ITEMS for a in gold_of(it, "gold_articles") if a not in NEW_BY_ART)
miss_hang = Counter(c for it in ITEMS for c in gold_of(it, "gold_hang") if c not in NEW_POS)
print("지금 데이터에 없는 정답 조문 id:", dict(miss_art) or "없음")
print("지금 데이터에 없는 정답 항 id:", dict(miss_hang) or "없음")

## 5. 검색 모델 · 색인

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음 (느려요)")

def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m

kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]

EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
EMBEDDER.max_seq_length = 1024
RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)

# 같은 글은 한 번만 임베딩 (① 조문 조각과 ② 항 조각이 겹치는 경우가 많음)
_TCACHE = {}
def embed_texts(texts):
    todo = [t for t in dict.fromkeys(texts) if t not in _TCACHE]
    if todo:
        for t, v in zip(todo, EMBEDDER.encode(todo, batch_size=32, normalize_embeddings=True,
                                               convert_to_numpy=True, show_progress_bar=True)):
            _TCACHE[t] = v.astype(np.float32)
    return np.stack([_TCACHE[t] for t in texts])

IDX = {}
for name, chunks in (("m1", M1C), ("all", NEWC)):
    t0 = time.time()
    IDX[name] = {"C": chunks, "BM25": BM25Okapi([ko_tokens(c["embed_text"]) for c in chunks]),
                 "EMB": embed_texts([c["embed_text"] for c in chunks])}
    print(f"{name}: {len(chunks)}조각 BM25 + 임베딩 {time.time() - t0:.0f}초")
NEW_EMB = IDX["all"]["EMB"]

In [ ]:
# ── ② 검색 · ③ 리랭커 (11/19 노트북과 같은 계산) ──
_QCACHE, _RCACHE = {}, {}
RR = {"pairs": 0, "sec": 0.0}

def embed_query(q):
    if q not in _QCACHE:
        _QCACHE[q] = EMBEDDER.encode([re.sub(r"\[[^\]]*\]", " ", q)], normalize_embeddings=True,
                                     convert_to_numpy=True)[0].astype(np.float32)
    return _QCACHE[q]

def candidates(name, question, n):
    ix = IDX[name]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    bs = ix["BM25"].get_scores(ko_tokens(bq))
    bm = [int(i) for i in np.argsort(-bs)[:max(CFG["k_bm25"], n)] if bs[i] > 0]
    de = [int(i) for i in np.argsort(-(ix["EMB"] @ embed_query(question)))[:max(CFG["k_dense"], n)]]
    score = defaultdict(float)
    for lst, w in ((bm, CFG["bm25_weight"]), (de, CFG["dense_weight"])):
        for r, i in enumerate(lst):
            score[i] += w / (CFG["rrf_c"] + r + 1)
    return sorted(score, key=score.get, reverse=True)[:n]

def rerank(question, js):
    # js: NEWC 번호들 → {번호: 점수 0~1}
    q = re.sub(r"\[[^\]]*\]", " ", question)
    todo = [j for j in dict.fromkeys(js) if (q, j) not in _RCACHE]
    if todo:
        t0 = time.perf_counter()
        s = np.asarray(RERANKER.predict([(q, NEWC[j]["embed_text"]) for j in todo], batch_size=16,
                                        convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
        if s.min() < 0 or s.max() > 1:
            s = 1 / (1 + np.exp(-s))
        _RCACHE.update({(q, j): float(v) for j, v in zip(todo, s)})
        RR["pairs"] += len(todo)
        RR["sec"] += time.perf_counter() - t0
    return {j: _RCACHE[(q, j)] for j in js}

def sims(question, js):
    v = embed_query(question)
    return {j: float(NEW_EMB[j] @ v) for j in js}

## ① 조 + 전결항목 1차 → 하위 항목 펼쳐 리랭크
```
1차: 조문 단위 조각 + 전결항목 → 후보 40조각 → 후보 단위로 묶음
     조문 조각 → 그 조 / 별표·서식안내 → 연결된 조(없으면 그대로) / 전결항목·부칙 → 그대로
인용: 상위 10개 조의 '따른다·준용' 대상 조(조당 3개, 임베딩 유사도 순) + '예외' 조(반대 방향) 추가, 한 단계만
펼치기: 조 → 항 + 연결 별표(별표당 3조각) + 연결 서식안내 / 따로 후보 → 그 조각들(별표면 3조각)
상한: 후보마다 가장 비슷한 조각 하나는 꼭 넣고, 나머지는 유사도 순으로 전체 200쌍까지
리랭크 1번 → 후보 점수 = 딸린 조각 중 최고점 → 순서
```

In [ ]:
def is_article(a): return a in ARTICLES

def units_m1(question):
    C1 = IDX["m1"]["C"]
    units, via = [], {}
    def add(u, how):
        if u not in NEW_BY_ART:
            return
        if u not in via:
            units.append(u); via[u] = how
        elif how not in via[u]:
            via[u] += "+" + how
    for i in candidates("m1", question, CFG["k_candidates"]):
        a = C1[i]["article_id"]
        if is_article(a):
            add(a, "1차")
        elif a in ART_OF_APPX:                      # 별표 → 연결된 조 (별표는 그 조에 딸려 들어감)
            for t in ART_OF_APPX[a][:2]:
                add(t, "1차(별표)")
        elif a in ART_OF_FORM:
            for t in ART_OF_FORM[a][:2]:
                add(t, "1차(서식)")
        else:
            add(a, "1차")                           # 전결항목·부칙·연결 없는 별표·서식
    # 인용된 조 (상위 10개 조에서만, 한 단계)
    top = [u for u in units if is_article(u)][:CFG["ref_top_units"]]
    for u in top:
        for kind, cands in (("인용", REF_FWD.get(u, [])), ("예외", EXC_REV.get(u, []))):
            cands = [t for t in cands if t in NEW_BY_ART]
            if not cands:
                continue
            best = sorted(cands, key=lambda t: -max(sims(question, NEW_BY_ART[t]).values()))
            for t in best[:CFG["ref_per_unit"]]:
                add(t, kind)
    return units, via

def expand(question, u):
    # 후보 하나 → (본문 조각, 딸린 조각) NEWC 번호
    body = list(NEW_BY_ART[u])
    attach = []
    if is_article(u):
        for x in APPX_OF.get(u, []):
            js = NEW_BY_ART.get(x, [])
            s = sims(question, js)
            attach += sorted(js, key=lambda j: -s[j])[:CFG["appx_chunks"]]
        for x in FORM_OF.get(u, []):
            attach += [j for j in NEW_BY_ART.get(x, []) if NEWC[j]["type"] == "서식안내"]
    elif NEWC[body[0]]["type"] == "별표":
        s = sims(question, body)
        body = sorted(body, key=lambda j: -s[j])[:CFG["appx_chunks"]]
    return body, attach

def search_m1(question):
    units, via = units_m1(question)
    parts = {u: expand(question, u) for u in units}
    every = dedupe(j for u in units for j in parts[u][0] + parts[u][1])
    s = sims(question, every)
    keep = set()
    for u in units:                                  # 후보마다 가장 비슷한 조각 하나는 꼭
        js = parts[u][0] + parts[u][1]
        keep.add(max(js, key=lambda j: s[j]))
    for j in sorted(every, key=lambda j: -s[j]):
        if len(keep) >= CFG["pair_cap"]:
            break
        keep.add(j)
    sc = rerank(question, [j for j in every if j in keep])
    ranked = []
    for u in units:
        b = [j for j in parts[u][0] if j in sc]
        a = [j for j in parts[u][1] if j in sc]
        if b or a:
            ranked.append({"id": u, "score": max(sc[j] for j in b + a), "body": b, "attach": a, "via": via[u]})
    ranked.sort(key=lambda r: -r["score"])
    return ranked, sc

## ② 모든 조각 1차 → 리랭크
```
1차: 항 단위 검색 조각 전부(항·별표·서식안내·부칙·전결항목) → 후보 K조각 (K = 40 / 200)
리랭크 → 조(article_id)로 묶음, 묶음 점수 = 최고점 → 순서. 펼치기·연결표 없음
```

In [ ]:
def search_m2(question, k):
    js = candidates("all", question, k)
    sc = rerank(question, js)
    groups = defaultdict(list)
    for j in js:
        groups[NEWC[j]["article_id"]].append(j)
    ranked = [{"id": a, "score": max(sc[j] for j in g), "body": g, "attach": [], "via": "1차"} for a, g in groups.items()]
    ranked.sort(key=lambda r: -r["score"])
    return ranked, sc

def head_of(c): return f"{c['article']}({c['article_title']})" if c["article_title"] else c["article"]

def context(ranked, sc, k=None):
    # 두 방식 같은 규칙: 본문은 점수 높은 조각부터 이어 붙여 1,200자, 딸린 별표 2조각·서식안내 1개
    out, seen = [], set()                            # seen: 같은 별표가 조 두 곳에 딸려도 한 번만 넣음
    for r in ranked[:k or CFG["k_context"]]:
        js = sorted(r["body"], key=lambda j: -sc[j])
        texts, ids, end = [], [], -1
        cap = CFG["max_chars_per_article"]
        for n_, j in enumerate(js):
            t, h = NEWC[j]["text"], head_of(NEWC[j])
            texts.append(t[len(h):].lstrip() if n_ and h and t.startswith(h) else t)
            end += len(texts[-1]) + 1
            if end <= cap:                               # 1,200자 안에 끝까지 들어간 조각만 '전달됨'
                ids.append(NEWC[j]["id"])
        text = "\n".join(texts)
        text = text if len(text) <= cap else text[:cap] + " …(생략)"
        covered = {r["id"]}
        att = sorted(r["attach"], key=lambda j: -sc[j])
        appx = [j for j in att if NEWC[j]["type"] != "서식안내" and j not in seen][:CFG["ctx_appx"]]
        form = [j for j in att if NEWC[j]["type"] == "서식안내" and j not in seen][:CFG["ctx_form"]]
        seen.update(appx + form)
        for j in appx + form:
            t = NEWC[j]["text"]
            if len(t) <= CFG["max_chars_appx"]:
                ids.append(NEWC[j]["id"])
            text += f"\n[{NEWC[j]['article']}]\n" + (t if len(t) <= CFG["max_chars_appx"] else t[:CFG["max_chars_appx"]] + " …(생략)")
            covered.add(NEWC[j]["article_id"])
        out.append({"id": r["id"], "text": text, "covered": covered, "ids": ids, "via": r["via"], "score": round(r["score"], 4)})
    return out

q = "며느리가 출산하면 할아버지인 저도 휴가 있어요?"
for label, fn in (("①", search_m1), ("② 40", lambda x: search_m2(x, CFG["k_all_small"]))):
    rk, sc = fn(q)
    print(label)
    for h in context(rk, sc, 3):
        print(f"  {h['score']:.3f} {h['id']} ({h['via']}) · {len(h['text'])}자 · {squash(h['text'])[:70]}")

## 6. 분류 없이 검색만 (세 방식 × 정답 조문 있는 문항 전부)
임직원은 실제 파이프라인도 검색어가 질문 그대로라, 분류가 '검색함'으로 보낸 문항은 이 결과를 그대로 써요. 문항마다 캐시를 비우고 처음부터 재요 (GPU 첫 호출은 예열로 뺌).

In [ ]:
def hang_sig(cid):
    c = NEWC[NEW_POS[cid]]
    body, h = c["text"], head_of(c)
    s = squash(body[len(h):] if h and body.startswith(h) else body)
    return (s[:30], s[-30:]) if len(s) > 30 else (s, s)

def delivered(cid, blob):
    if cid not in NEW_POS:
        return False
    a, b = hang_sig(cid)
    return a in blob and b in blob

METHODS = {"① 조+전결 → 펼치기": search_m1,
           "② 모든 조각 (40)": lambda q: search_m2(q, CFG["k_all_small"])}
QS = [it for it in ITEMS if SPLIT[it["id"]] in ("dev", "test") and gold_of(it, "gold_hang")]
print("정답 조각 있는 문항", len(QS), "· 확인 안 된 정답 항", sum((it.get("gold_hang") or {}).get("auto") is False for it in QS))

def score_one(it, fn):
    _QCACHE.clear(); _RCACHE.clear(); RR.update(pairs=0, sec=0.0)
    if HAS_GPU: torch.cuda.synchronize()
    tq = time.perf_counter()
    ranked, sc = fn(it["question"])
    ctx = context(ranked, sc)
    if HAS_GPU: torch.cuda.synchronize()
    ms = (time.perf_counter() - tq) * 1000
    blob = squash("\n".join(h["text"] for h in ctx))
    gold = set(gold_of(it, "gold_hang"))                          # 정답 조각 id (항·별표 조각)
    arts = gold_of(it, "gold_articles")
    sent = {c for h in ctx for c in h["ids"]}                      # LLM 입력에 온전히 들어간 조각
    chunk_rank = [NEWC[j]["id"] for j in sorted(sc, key=lambda j: -sc[j])]   # 리랭커 점수 순 조각
    first = next((n + 1 for n, c in enumerate(chunk_rank) if c in gold), None)
    cov5 = set().union(*[h["covered"] for h in ctx]) if ctx else set()
    hit_ctx = [h for h in ctx if h["covered"] & set(arts)]
    return {"r5": len(gold & sent) / len(gold),
            "r20": len(gold & set(chunk_rank[:20])) / len(gold),
            "mrr": 1 / first if first else 0.0,
            "art5": len(set(arts) & cov5) / len(arts) if arts else None,
            "ref_only": bool(hit_ctx) and all(h["via"] in ("인용", "예외") for h in hit_ctx),
            "ctx_chars": sum(len(h["text"]) for h in ctx), "ms": ms, "pairs": RR["pairs"], "rerank_ms": RR["sec"] * 1000,
            "ctx": [f"{h['id']}({h['via']})" for h in ctx]}

BASE = {}
for label, fn in METHODS.items():
    fn("연차는 며칠이야?")                          # 예열
    t0 = time.time()
    BASE[label] = {it["id"]: score_one(it, fn) for it in QS}
    print(f"{label}: {len(QS)}문항 {time.time() - t0:.0f}초")
(OUT_DIR / "search_base.json").write_text(json.dumps(BASE, ensure_ascii=False), encoding="utf-8")

## 7. vLLM 띄우기 (27B 다운로드가 끝날 때까지 기다려요)

검색 모델(BGE-M3·리랭커)은 GPU에 그대로 두고, vLLM은 `--gpu-memory-utilization 0.75`로 띄워요.

In [ ]:
import urllib.request
LLM_LOG = Path("/content/vllm.log")
LLM_API_KEY = hashlib.sha256(os.urandom(16)).hexdigest()[:32]
LLM_BASE_URL = f"http://127.0.0.1:{CFG['llm_port']}/v1"

while "SETUP_DONE" not in SETUP_LOG.read_text():
    if "SETUP_PROC" in globals() and SETUP_PROC.poll() not in (None, 0):
        print(SETUP_LOG.read_text()[-3000:]); raise RuntimeError("vLLM 설치·다운로드 실패 (로그 위)")
    last = (SETUP_LOG.read_text().strip().splitlines() or ["시작 전"])[-1]
    print("설치·다운로드 기다리는 중…", last[:100]); time.sleep(30)

subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(3)
help_text = subprocess.run([str(VLLM_ENV / "bin/vllm"), "serve", "--help=all"], capture_output=True, text=True).stdout
cmd = [str(VLLM_ENV / "bin/vllm"), "serve", CFG["llm_model"], "--served-model-name", CFG["served_name"],
       "--host", "127.0.0.1", "--port", str(CFG["llm_port"]), "--api-key", LLM_API_KEY,
       "--max-model-len", str(CFG["max_model_len"]), "--gpu-memory-utilization", str(CFG["vllm_gpu_util"]),
       "--max-num-seqs", "16",           # Qwen3.5 Mamba 캐시 때문에 기본 256이면 A100 40GB에서 시작이 안 돼요
       "--reasoning-parser", "qwen3", "--trust-remote-code", "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
if "--default-chat-template-kwargs" in help_text:
    cmd += ["--default-chat-template-kwargs", json.dumps({"enable_thinking": False})]
if N_GPU > 1:
    cmd += ["--tensor-parallel-size", str(N_GPU)]
print(" ".join(cmd).replace(LLM_API_KEY, "***"))
LLM_PROC = subprocess.Popen(cmd, stdout=open(LLM_LOG, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {})})
t0 = time.time()
while True:
    try:
        req = urllib.request.Request(f"{LLM_BASE_URL}/models", headers={"Authorization": f"Bearer {LLM_API_KEY}"})
        json.loads(urllib.request.urlopen(req, timeout=5).read())
        print(f"✅ vLLM 준비 완료 ({time.time() - t0:.0f}초)"); break
    except Exception:
        if LLM_PROC.poll() is not None:
            log = LLM_LOG.read_text(errors="replace").splitlines()
            keys = ("out of memory", "OutOfMemory", "Error:", "ValueError", "not supported", "No available memory", "max_model_len", "CUDA")
            print("── 원인 후보 ──"); print("\n".join([l for l in log if any(k in l for k in keys)][-25:]) or "\n".join(log[-40:]))
            raise RuntimeError(f"vLLM이 시작하지 못했어요 (전체 로그: {LLM_LOG})")
        time.sleep(15)

## 8. backend 분류 LLM을 vLLM에 연결 · 실패 세기

backend는 분류 LLM을 `llm_classify.chat_json`으로 불러요. 이 함수 하나만 vLLM(OpenAI 형식)으로 바꾸고 **프롬프트·스키마·응답 검증·규칙 대체는 backend 그대로**예요.

In [ ]:
CALLS = {"n": 0, "fail": 0, "bad_json": 0, "ms": [], "last_error": ""}
FALLBACK = Counter()      # 단계별 규칙 대체 횟수
CUR = {"calls": 0, "fail": 0, "fallback": []}     # 지금 분류 중인 문항 하나의 기록


def _vllm_chat_json(system, user, schema, num_predict=256):
    from openai import OpenAI
    global _LLM, _LLM_KEY
    # vLLM 셀을 다시 돌리면 API 키가 새로 생겨요 → 예전 키로 만든 클라이언트를 쓰면 401. 키가 바뀌면 새로 만듦
    if globals().get("_LLM_KEY") != (LLM_BASE_URL, LLM_API_KEY):
        _LLM = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=300, max_retries=2)
        _LLM_KEY = (LLM_BASE_URL, LLM_API_KEY)
    r = _LLM.chat.completions.create(
        model=CFG["served_name"],
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0, max_tokens=num_predict,
        response_format={"type": "json_schema", "json_schema": {"name": "out", "schema": schema}},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    return be_nodes._clean(r.choices[0].message.content or "")


def counted_chat_json(transport):
    # chat_json을 감싸서 호출·실패를 셈. 실패하면 backend가 잡아 규칙 결과로 대체하도록 ValueError를 올림
    def chat_json(system, user, schema, num_predict=256):
        CALLS["n"] += 1; CUR["calls"] += 1
        t = time.perf_counter()
        try:
            out = transport(system, user, schema, num_predict)
        except Exception as e:
            CALLS["fail"] += 1; CUR["fail"] += 1
            CALLS["last_error"] = f"{type(e).__name__}: {str(e)[:300]}"
            raise ValueError("분류 LLM 요청 실패")
        finally:
            CALLS["ms"].append((time.perf_counter() - t) * 1000)
        try:
            json.loads(out)
        except (json.JSONDecodeError, TypeError):
            CALLS["bad_json"] += 1; CUR["fail"] += 1
        return out
    return chat_json


def count_fallback(module, name, label):
    orig = getattr(module, name)
    if getattr(orig, "_counted", False):      # 셀을 다시 실행해도 두 번 감싸지 않게
        return
    def wrapped(*a, **kw):
        r = orig(*a, **kw)
        if r is None:
            FALLBACK[label] += 1; CUR["fallback"].append(label)
        return r
    wrapped._counted = True
    setattr(module, name, wrapped)


def check_llm(where):
    # LLM이 전부 실패하면 분류가 규칙으로만 돌아 결과가 틀려요 → 바로 멈춤
    if CALLS["n"] and CALLS["fail"] == CALLS["n"]:
        alive = "LLM_PROC" in globals() and LLM_PROC.poll() is None
        tail = "\n".join(LLM_LOG.read_text(errors="replace").splitlines()[-15:]) if LLM_LOG.exists() else ""
        raise RuntimeError(f"{where}: 분류 LLM 호출 {CALLS['n']}번 모두 실패 ({CALLS['last_error']})\n"
                           f"vLLM 프로세스 {'살아 있음' if alive else '죽음 → 7. vLLM 시작 셀부터 다시'}\n{tail}")


def connect_llm(transport):
    llm_classify.chat_json = counted_chat_json(transport)
    be_config.CLASSIFY_LLM = True           # 분류 보조 LLM 켜기 (enabled() = CLASSIFY_LLM and not USE_FAKE_LLM)
    be_config.USE_FAKE_LLM = False
    be_config.CLASSIFY_MODEL = CFG["served_name"]
    assert llm_classify.enabled() and llm_classify.is_local()   # 보안문서 LLM 판단은 로컬 LLM일 때만 켜져요
    count_fallback(llm_classify, "classify", "임직원 의도")
    count_fallback(admin_intent_llm, "classify", "인사팀 의도")
    count_fallback(admin_params_llm, "extract", "인사팀 인자")
    count_fallback(conf_doc, "_llm_detect", "보안문서")

In [ ]:
connect_llm(_vllm_chat_json)
s = {"employee_id": CFG["employee_login"], "role": "employee", "question": "결혼하면 휴가 며칠이야?"}
s |= be_nodes.guard(s); t = time.time(); s |= be_nodes.classify(s)
print(f"{time.time() - t:5.1f}초 | 결혼하면 휴가 며칠이야? → {s['intent']}")
print(CALLS["n"], "호출,", CALLS["fail"], "실패")
check_llm("연결 시험")

## 9. 분류 → 검색으로 가나? (dev·test 전 문항, 정답 없는 문항도)

분기는 backend `app/graph/build.py`의 `run_pipeline`과 같은 순서예요.

In [ ]:
def route(s):
    # build.run_pipeline 분기 그대로 → (검색하나, 이유)
    if s["intent"] == "sensitive":
        return False, "sensitive"
    if s.get("refuse_reason") == "chat_log":
        return False, "refuse(대화 기록)"
    if s.get("conf_doc"):
        return False, "기밀문서"
    if s["intent"] == "refuse":
        return False, "refuse"
    if s["intent"] == "no_question" or not be_nodes.has_question_content(s["masked_question"]):
        return False, "no_question"
    if s["intent"] == "out_of_scope":
        return False, "out_of_scope"
    if s["intent"] == "lookup":
        return False, "lookup"
    return True, s["intent"]          # rule · both (인사팀 data도 both로 봄)


def classify_item(x):
    s = {"employee_id": x.get("emp_id") or CFG["employee_login"], "role": "employee", "question": x["question"]}
    CUR.update(calls=0, fail=0, fallback=[])
    t = time.perf_counter()
    s |= be_nodes.guard(s)
    s |= be_nodes.classify(s)
    ms = (time.perf_counter() - t) * 1000
    searched, reason = route(s)
    hint = s.get("search_hint") or ""
    return {"intent": s["intent"], "conf_doc": s.get("conf_doc"), "admin_tool": s.get("admin_tool"),
            "search_hint": hint, "searched": searched, "reason": reason,
            "query": f"{x['question']} {hint}" if hint else x["question"],      # nodes.retrieve와 같은 검색어
            "classify_ms": round(ms, 1), "llm_calls": CUR["calls"], "llm_fail": CUR["fail"], "fallback": list(CUR["fallback"])}


CLASSIFY_TODAY = date.today().isoformat()       # 분류 LLM 기준일 (date.today(), 코드 안 고침)
CALLS.update(n=0, fail=0, bad_json=0, ms=[]); FALLBACK.clear()
EVAL = [it for it in ITEMS if SPLIT[it["id"]] in ("dev", "test")]
CLS, t0 = {}, time.time()
for n, it in enumerate(EVAL, 1):
    CLS[it["id"]] = classify_item(it)
    if n in (5, 50) or n == len(EVAL):
        check_llm(f"분류 {n}문항째")
    if n % 50 == 0:
        print(f"{n}/{len(EVAL)} {time.time() - t0:.0f}초")
(OUT_DIR / "classify.json").write_text(json.dumps(CLS, ensure_ascii=False), encoding="utf-8")
print(f"LLM 호출 {CALLS['n']}번 · 실패 {CALLS['fail']}번 · 규칙 대체 {dict(FALLBACK)}")
print(Counter(c["reason"] for c in CLS.values()))

## 10. 결과

In [ ]:
def _mean(v):
    v = [x for x in v if x is not None]
    return round(float(np.mean(v)), 3) if v else None

def summary(split):
    ids = [it["id"] for it in QS if split == "dev+test" or SPLIT[it["id"]] == split]
    srch = [i for i in ids if CLS[i]["searched"]]
    out = {}
    for label in METHODS:
        B = BASE[label]
        stage2 = lambda k: _mean([(B[i][k] if CLS[i]["searched"] else (0.0 if B[i][k] is not None else None)) for i in ids])
        out[label] = {
            "문항(정답 있음)": len(ids), "검색 누락": len(ids) - len(srch), "검색 누락률": round(1 - len(srch) / len(ids), 3),
            "2단계 종합 정답 조각 recall@5": stage2("r5"),
            "분류 없이 검색만 조각 recall@5": _mean([B[i]["r5"] for i in ids]),
            "검색한 문항 조각 recall@5": _mean([B[i]["r5"] for i in srch]),
            "검색한 문항 조각 recall@20": _mean([B[i]["r20"] for i in srch]),
            "검색한 문항 조각 MRR": _mean([B[i]["mrr"] for i in srch]),
            "(참고) 검색한 문항 조문 recall@5": _mean([B[i]["art5"] for i in srch]),
            "인용·예외로만 살린 문항": sum(B[i]["ref_only"] for i in srch),
            "LLM 입력 글자수": _mean([B[i]["ctx_chars"] for i in srch]),
            "리랭커 쌍 평균": _mean([B[i]["pairs"] for i in srch]), "리랭커 쌍 최대": max([B[i]["pairs"] for i in srch], default=None),
            "검색 p50 ms": round(float(np.median([B[i]["ms"] for i in srch])), 1) if srch else None,
            "검색 p95 ms": round(float(np.percentile([B[i]["ms"] for i in srch], 95)), 1) if srch else None,
        }
    return pd.DataFrame(out)

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음", "· 분류", CFG["llm_model"])
cms = [c["classify_ms"] for c in CLS.values()]
print(f"분류 p50 {np.median(cms):.0f}ms · p95 {np.percentile(cms, 95):.0f}ms · LLM 호출 {CALLS['n']} · 실패 {CALLS['fail']} · 규칙 대체 {dict(FALLBACK)}")
SUMS = {}
for split in ("dev", "test", "dev+test"):
    print(f"\n========== {split} ==========")
    SUMS[split] = summary(split)
    display(SUMS[split])

print("\n■ 정답이 있는데 검색을 안 한 문항")
display(pd.DataFrame([{"split": SPLIT[it["id"]], "id": it["id"], "question": it["question"], "intent": CLS[it["id"]]["intent"],
                       "이유": CLS[it["id"]]["reason"]} for it in QS if not CLS[it["id"]]["searched"]]))
print("■ 정답이 없는데 검색한 문항 수:", sum(CLS[it["id"]]["searched"] for it in EVAL if not gold_of(it, "gold_articles")),
      "/", sum(not gold_of(it, "gold_articles") for it in EVAL))

## 11. 문항별로 갈린 것 (검색한 문항, ① vs ② 200)

In [ ]:
BY_ID = {it["id"]: it for it in QS}
def diff(a, b, key):
    d = {i: (BASE[a][i][key], BASE[b][i][key]) for i in BY_ID if CLS[i]["searched"]
         and BASE[a][i][key] is not None and BASE[b][i][key] is not None and BASE[a][i][key] != BASE[b][i][key]}
    win = [i for i, (x, y) in d.items() if x > y]; lose = [i for i, (x, y) in d.items() if x < y]
    print(f"■ {key}: {a} 이 나음 {len(win)}문항 · {b} 이 나음 {len(lose)}문항")
    for ids, who in ((win, a), (lose, b)):
        for i in ids[:15]:
            print(f"  [{who[:1]}] {i} {BY_ID[i]['question'][:45]}")
            print(f"      정답 조각 {gold_of(BY_ID[i], 'gold_hang')}")
            print(f"      ① {BASE[a][i]['ctx']}")
            print(f"      ② {BASE[b][i]['ctx']}")

A_, B_ = "① 조+전결 → 펼치기", "② 모든 조각 (40)"
for key in ("r5", "mrr"):
    diff(A_, B_, key); print()
print("■ ①에서 인용·예외로만 정답이 들어온 문항")
display(pd.DataFrame([{"id": i, "question": BY_ID[i]["question"], "gold": gold_of(BY_ID[i], "gold_articles"), "ctx": BASE[A_][i]["ctx"]}
                      for i in BY_ID if CLS[i]["searched"] and BASE[A_][i]["ref_only"]]))

## 12. 저장 (zip 내려받기 · Drive)

In [ ]:
ROWS = []
for it in QS:
    c = CLS[it["id"]]
    for label in METHODS:
        ROWS.append({"split": SPLIT[it["id"]], "id": it["id"], "question": it["question"], "방식": label,
                     "intent": c["intent"], "searched": c["searched"], "reason": c["reason"],
                     "gold_articles": gold_of(it, "gold_articles"), **BASE[label][it["id"]]})
META = {"commits": COMMITS, "gpu": f"{GPU_NAME} × {N_GPU}", "classify_llm": CFG["llm_model"], "classify_today": CLASSIFY_TODAY,
        "cfg": CFG, "llm_calls": CALLS["n"], "llm_fail": CALLS["fail"], "rule_fallback": dict(FALLBACK),
        "run_at": datetime.now().isoformat(timespec="seconds")}
(OUT_DIR / "rows.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in ROWS), encoding="utf-8")
(OUT_DIR / "summary.json").write_text(json.dumps({"meta": META, **{k: v.to_dict() for k, v in SUMS.items()}},
                                                 ensure_ascii=False, indent=1, default=str), encoding="utf-8")
for k, v in SUMS.items():
    v.to_csv(OUT_DIR / f"summary_{k}.csv", encoding="utf-8-sig")
ZIP = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("저장:", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/hr_chatbot_eval"); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(ZIP, dst); print("Drive에도 저장:", dst / Path(ZIP).name)
except Exception as e:
    print(f"내려받기·Drive 건너뜀 ({type(e).__name__})")